In [90]:
# ==========================================================
# TruthLens AI
# Model Training Notebook
# Part-1
# ==========================================================

import os
import re
import pickle
import warnings
import nltk
import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# ----------------------------------------------------------
# Download Stopwords
# ----------------------------------------------------------

try:
    stopwords.words("english")
except:
    nltk.download("stopwords")

print("="*60)
print("TruthLens AI")
print("Fake News Detection Model Training")
print("="*60)

# ----------------------------------------------------------
# Load Dataset
# ----------------------------------------------------------

fake = pd.read_csv("dataset/Fake.csv")
true = pd.read_csv("dataset/True.csv")

print("\nDataset Loaded Successfully")

print("Fake Shape :", fake.shape)
print("True Shape :", true.shape)

# ----------------------------------------------------------
# Create Labels
# ----------------------------------------------------------

fake["label"] = 1
true["label"] = 0

# ----------------------------------------------------------
# Merge Dataset
# ----------------------------------------------------------

df = pd.concat([fake, true], axis=0)

# Shuffle Dataset

df = df.sample(frac=1, random_state=42)

df.reset_index(drop=True, inplace=True)

# Keep only required columns

df = df[["text", "label"]]

print("\nMerged Dataset Shape :", df.shape)

print("\nLabel Distribution")

print(df["label"].value_counts())

# ----------------------------------------------------------
# NLP Preprocessing
# ----------------------------------------------------------

stemmer = PorterStemmer()

STOP_WORDS = set(stopwords.words("english"))

def preprocess(text):

    text = str(text)

    text = re.sub(r"[^a-zA-Z]", " ", text)

    text = text.lower()

    words = text.split()

    words = [
        stemmer.stem(word)
        for word in words
        if word not in STOP_WORDS
    ]

    return " ".join(words)

print("\nCleaning Articles...")

df["text"] = df["text"].apply(preprocess)

print("Text Preprocessing Completed")

# ----------------------------------------------------------
# Features & Labels
# ----------------------------------------------------------

X = df["text"]

y = df["label"]

print("\nTotal Samples :", len(X))

print("\nFirst Sample")

print(X.iloc[0][:400])
# ==========================================================
# Train-Test Split
# ==========================================================

print("\nSplitting Dataset...")

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples :", x_train.shape[0])
print("Testing Samples  :", x_test.shape[0])

# ==========================================================
# TF-IDF Vectorization
# ==========================================================

print("\nApplying TF-IDF Vectorization...")

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

print("Vectorization Completed")

print("Training Matrix Shape :", x_train.shape)
print("Testing Matrix Shape  :", x_test.shape)

# ==========================================================
# Logistic Regression Model
# ==========================================================

print("\nTraining Logistic Regression Model...")

model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

model.fit(x_train, y_train)

print("Model Training Completed Successfully.")

# ==========================================================
# Prediction
# ==========================================================

print("\nGenerating Predictions...")

y_pred = model.predict(x_test)

print("Prediction Completed.")

# ==========================================================
# Accuracy
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n" + "="*50)
print("MODEL PERFORMANCE")
print("="*50)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
# ==========================================================
# MODEL EVALUATION
# ==========================================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)

print(classification_report(y_test, y_pred))

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)

cm = confusion_matrix(y_test, y_pred)

print(cm)

# ==========================================================
# SAVE MODEL
# ==========================================================

print("\nSaving Model...")

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("model.pkl Saved Successfully")

print("\nSaving Vectorizer...")

with open("vector.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("vector.pkl Saved Successfully")

# ==========================================================
# LOAD MODEL AGAIN
# ==========================================================

print("\nLoading Saved Files...")

with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("vector.pkl", "rb") as f:
    loaded_vectorizer = pickle.load(f)

print("Files Loaded Successfully")

# ==========================================================
# MANUAL TESTING FUNCTION
# ==========================================================

def predict_news(news):

    news = preprocess(news)

    vector = loaded_vectorizer.transform([news])

    prediction = loaded_model.predict(vector)[0]

    if hasattr(loaded_model, "predict_proba"):

        probability = loaded_model.predict_proba(vector)[0]

        confidence = max(probability) * 100

    else:

        confidence = None

    if prediction == 0:
        result = "✅ REAL NEWS"
    else:
        result = "❌ FAKE NEWS"

    return result, confidence

# ==========================================================
# SAMPLE TESTS
# ==========================================================

real_news = """
WASHINGTON (Reuters) - The Federal Reserve announced that it
will continue monitoring inflation before making future
interest rate decisions.
"""

fake_news = """
Scientists officially confirmed that drinking hot salt water
every morning cures every disease within three days.
"""

print("\n" + "="*60)
print("TESTING MODEL")
print("="*60)

result, confidence = predict_news(real_news)

print("\nReal News Test")
print(result)

if confidence is not None:
    print("Confidence : {:.2f}%".format(confidence))

result, confidence = predict_news(fake_news)

print("\nFake News Test")
print(result)

if confidence is not None:
    print("Confidence : {:.2f}%".format(confidence))

print("\n" + "="*60)
print("TruthLens AI Model Training Completed Successfully")
print("="*60)

TruthLens AI
Fake News Detection Model Training

Dataset Loaded Successfully
Fake Shape : (23481, 4)
True Shape : (21417, 4)

Merged Dataset Shape : (44898, 2)

Label Distribution
label
1    23481
0    21417
Name: count, dtype: int64

Cleaning Articles...
Text Preprocessing Completed

Total Samples : 44898

First Sample
st centuri wire say ben stein reput professor pepperdin univers also hollywood fame appear tv show film ferri bueller day made provoc statement judg jeanin pirro show recent discuss halt impos presid trump execut order travel stein refer judgement th circuit court washington state coup tat execut branch constitut stein went call judg seattl polit puppet judiciari polit pawn watch interview complet

Splitting Dataset...
Training Samples : 35918
Testing Samples  : 8980

Applying TF-IDF Vectorization...
Vectorization Completed
Training Matrix Shape : (35918, 5000)
Testing Matrix Shape  : (8980, 5000)

Training Logistic Regression Model...
Model Training Completed Successfu